# Test of using the Open Food Facts API

In [5]:
%pip install requests

  Obtaining dependency information for requests from https://files.pythonhosted.org/packages/70/8e/0e2d847013cb52cd35b38c009bb167a1a26b2ce6cd6965bf26b47bc0bf44/requests-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for charset-normalizer<4,>=2 from https://files.pythonhosted.org/packages/b6/7c/8debebb4f90174074b827c63242c23851bdf00a532489fba57fef3416e40/charset_normalizer-3.3.2-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for idna<4,>=2.5 from https://files.pythonhosted.org/packages/c2/e7/a82b05cf63a603df6e68d59ae6a68bf5064484a0718ea5033660af4b54a9/idna-3.6-py3-none-any.whl.metadata
  Obtaining dependency information for urllib3<3,>=1.21.1 from https://files.pythonhosted.org/packages/96/94/c31f58c7a7f470d5665935262ebd7455c7e4c7782eb525658d3dbf4b9403/urllib3-2.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for certifi>=2017.4.17 from https://files.pythonhosted.org/packages/64/62/428ef076be88fa93716b576e4a01f919d25968913e81

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 23.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Barcode lookup

In [2]:

import requests
from Food import Food

# Replace this with the barcode you want to lookup
barcode = " 4053400205298 "

# Construct the API URL
url = f"https://world.openfoodfacts.org/api/v0/product/{barcode}.json"
# Send a GET request to the API
response = requests.get(url)
# Check if the request was successful
if response.status_code == 200:
    # Extract the nutrition information from the response JSON
    data = response.json()
    if "product" in data:
        product = data["product"]
        if "nutriments" in product:
            n = {key.replace("_100g",""): value for key, value in product["nutriments"].items() if "_100g" in key}
            ret = Food(n,100)
            print(n)
            print(f"{Food.valid_name(product['product_name'])} = Food({ret.nutrition_data})")
else:
    print(f"Error {response.status_code}: {response.reason}")



{}
landbier = Food({})


## Product search

In [1]:
import requests
from IPython.display import Image, display

# Replace this with the name of the product you want to search for
product_name = "Altenmünster "

# Construct the API URL for the search
url = f"https://world.openfoodfacts.org/cgi/search.pl?search_terms={product_name}&search_simple=1&action=process&json=1"

# Send a GET request to the API
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Extract the search results from the response JSON
    data = response.json()
    if "products" in data:
        products = data["products"]
        # Loop over the search results and print information about each product
        for product in products:
            print(f"Product name: {product.get('product_name')}")
            print(f"Barcode: {product.get('code')}")
            """
            print(f"Brand: {product.get('brands')}")
            print(f"Categories: {product.get('categories')}")

            # Print the nutrition information for the product
            print("Nutrition information:")
            nutriments = product.get("nutriments", {})
            for nutriment_name, nutriment_value in nutriments.items():
                print(f"  {nutriment_name}: {nutriment_value}")

            print(f"Ingredients: {product.get('ingredients_text')}")
            print(f"Allergens: {product.get('allergens')}")
            print(f"Additives: {product.get('additives_tags')}")
            print(f"Image URL: {product.get('image_url')}")
            print(f"Product URL: {product.get('url')}")
            """
            if product.get('image_url'):
                display(Image(url=product.get('image_url')))
            print()
else:
    print(f"Error {response.status_code}: {response.reason}")



In [ ]:
import csv

def print_first_x_lines(filename,x=20):
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)  # read and store the header row
        print(header)  # print the header row
        
        for i, row in enumerate(reader):
            if i < x:
                print(row)  # print the current row
            else:
                break  # exit the loop after 100 rows

print_first_x_lines('en.openfoodfacts.org.products.csv')

## Reading chunk wise with Pandas

does **NOT** work yet

In [38]:
!pip install pandas

import pandas as pd

def search_csv_file(file_path, term):
    # Define the chunk size (number of rows to read at once)
    chunk_size = 100000
    chunk_counter = 0
    # Open the CSV file in chunks and iterate through them
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, sep = '\t'):
        chunk_counter += 1
        print(chunk_counter)
        # Filter the rows that contain the search term in the specified columns
        filtered_chunk = chunk[(chunk['product_name'].str.contains(term, na=False)) | (chunk['generic_name'].str.contains(term, na=False))]
        
        # Print the filtered rows
        for index, row in filtered_chunk.iterrows():
            print(row)

search_csv_file('en.openfoodfacts.org.products.csv', 'roullade')

C:\Users\kaiuw\AppData\Local\Temp\ipykernel_12340\573122892.py:10: DtypeWarning: Columns (0,9,15,29,30,31,33,70) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size, sep = '\t'):


1


C:\Users\kaiuw\AppData\Local\Temp\ipykernel_12340\573122892.py:10: DtypeWarning: Columns (0,9,15,21,22,23,24,25,29,30,31,33,70) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size, sep = '\t'):


2


C:\Users\kaiuw\AppData\Local\Temp\ipykernel_12340\573122892.py:10: DtypeWarning: Columns (0,15,29,30,31,33) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size, sep = '\t'):


...

27


C:\Users\kaiuw\AppData\Local\Temp\ipykernel_12340\573122892.py:10: DtypeWarning: Columns (0,9,31,33,65,70) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size, sep = '\t'):


28


C:\Users\kaiuw\AppData\Local\Temp\ipykernel_12340\573122892.py:10: DtypeWarning: Columns (0,9,31,33,65,70) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size, sep = '\t'):


29
